# Notebook 05 — Real-Time ASL Detection
## American Sign Language (ASL) Detection
---
Loads the trained model, opens the webcam, and detects ASL signs in real time.

**How it works:**
1. Capture webcam frame
2. Extract 126 hand landmarks via MediaPipe
3. Buffer last 30 frames in a rolling window
4. Feed the window to the BiLSTM every frame
5. Average predictions over the last 10 frames to reduce flickering
6. Display the predicted word on screen

**Controls:** `A` add word to sentence &nbsp;|&nbsp; `C` clear sentence &nbsp;|&nbsp; `Q` quit

In [7]:
import os, json, time, collections
import numpy as np
import cv2
import mediapipe as mp
import warnings
warnings.filterwarnings('ignore')
from tensorflow.keras.models import load_model

print(f'OpenCV {cv2.__version__}  |  MediaPipe {mp.__version__}')

OpenCV 4.13.0  |  MediaPipe 0.10.35


In [8]:
# Load model and configuration
model = load_model('models/best_lstm_model.keras')
print('Model loaded.')

with open('models/model_metadata.json') as f:
    meta = json.load(f)
with open('processed_data/sequence_config.json') as f:
    cfg = json.load(f)

CLASS_NAMES = meta['class_names']
SEQ_LEN     = cfg['SEQUENCE_LENGTH']
FEAT_SIZE   = cfg['FEATURE_SIZE']
THRESHOLD   = meta.get('confidence_threshold', 0.7)

print(f'Classes   : {len(CLASS_NAMES)}  {CLASS_NAMES}')
print(f'Seq length: {SEQ_LEN}  |  Features: {FEAT_SIZE}  |  Threshold: {THRESHOLD}')

Model loaded.
Classes   : 30  ['ANGRY', 'ART', 'BAD', 'COME', 'EXCITED', 'FAMILY', 'FRIEND', 'GO', 'GOOD', 'HAPPY', 'HELLO', 'HELP', 'HOME', 'HURT', 'KNOW', 'LOVE', 'MORE', 'NO', 'PLEASE', 'SAD', 'SCARED', 'SORRY', 'STOP', 'THANK YOU', 'THINK', 'TIME', 'TIRED', 'WATER', 'WORK', 'YES']
Seq length: 30  |  Features: 126  |  Threshold: 0.7


In [9]:
# Set up MediaPipe HandLandmarker for real-time video tracking
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions
from mediapipe.tasks.python.vision.hand_landmarker import HandLandmarksConnections
from mediapipe.tasks.python.vision import drawing_utils  as mp_drawing
from mediapipe.tasks.python.vision import drawing_styles as mp_draw_styles
from mediapipe.tasks.python.core.base_options import BaseOptions
from mediapipe.tasks.python.vision.core.vision_task_running_mode import VisionTaskRunningMode

HAND_CONN = HandLandmarksConnections.HAND_CONNECTIONS

def make_landmarker():
    opts = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
        running_mode=VisionTaskRunningMode.VIDEO,
        num_hands=2, min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5, min_tracking_confidence=0.5,
    )
    return HandLandmarker.create_from_options(opts)

def get_landmarks(frame, landmarker, ts_ms):
    """Extract hand landmarks from a webcam frame. Returns (lm_vector, hand_found, annotated_frame)."""
    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts_ms)
    lm, found, ann = np.zeros(FEAT_SIZE, np.float32), False, frame.copy()
    if result.hand_landmarks:
        found = True
        for hi, hand in enumerate(result.hand_landmarks[:2]):
            mp_drawing.draw_landmarks(ann, hand, HAND_CONN,
                mp_draw_styles.get_default_hand_landmarks_style(),
                mp_draw_styles.get_default_hand_connections_style())
            off = hi * 63
            for li, l in enumerate(hand):
                lm[off + li*3], lm[off + li*3+1], lm[off + li*3+2] = l.x, l.y, l.z
    return lm, found, ann

print('MediaPipe ready.')

MediaPipe ready.


In [10]:
# Main real-time detection loop
def run_detector(camera_id=0):
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print(f'Cannot open camera {camera_id}. Try camera_id=1.')
        return
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    buffer    = collections.deque(maxlen=SEQ_LEN)    # rolling 30-frame window
    prob_hist = collections.deque(maxlen=10)          # last 10 predictions for smoothing
    sentence  = []
    fps_q     = collections.deque(maxlen=30)
    t0        = time.time()
    t0_ms     = int(t0 * 1000)
    prev_t    = t0
    last_hand = False
    pred_word, pred_conf = None, 0.0

    print('Webcam open. Press A:Add  C:Clear  Q:Quit')

    with make_landmarker() as lm_detector:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.flip(frame, 1)
            ts_ms = int(time.time() * 1000) - t0_ms

            lm_vec, hand_found, ann = get_landmarks(frame, lm_detector, ts_ms)

            # Clear buffer when hand leaves frame
            if last_hand and not hand_found:
                buffer.clear()
                prob_hist.clear()
            last_hand = hand_found
            buffer.append(lm_vec)

            # Predict once buffer is full (30 frames)
            if len(buffer) == SEQ_LEN:
                seq   = np.expand_dims(np.array(buffer, np.float32), 0)
                probs = model.predict(seq, verbose=0)[0]
                prob_hist.append(probs)
                avg   = np.mean(list(prob_hist), axis=0)  # average over last 10 predictions
                idx   = int(np.argmax(avg))
                pred_word, pred_conf = CLASS_NAMES[idx], float(avg[idx])

            # FPS counter
            now = time.time()
            fps_q.append(1.0 / max(now - prev_t, 1e-6))
            prev_t = now
            fps    = np.mean(fps_q)

            # Draw UI overlay
            h, w = ann.shape[:2]
            cv2.rectangle(ann, (0, 0), (w, 60), (0, 0, 0), -1)
            color = (0, 255, 0) if hand_found else (0, 0, 255)
            cv2.putText(ann, 'HAND ON' if hand_found else 'HAND OFF',
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.putText(ann, f'FPS: {fps:.1f}',
                        (w - 120, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            if pred_word and pred_conf >= THRESHOLD:
                cv2.putText(ann, f'{pred_word}  {pred_conf*100:.0f}%',
                            (10, h - 60), cv2.FONT_HERSHEY_DUPLEX, 1.5, (0, 255, 100), 3)
            if sentence:
                cv2.putText(ann, 'Sentence: ' + ' '.join(sentence[-6:]),
                            (10, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(ann, 'A:Add  C:Clear  Q:Quit',
                        (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

            cv2.imshow('ASL Real-Time Detector', ann)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27):
                break
            elif key == ord('a') and pred_word and pred_conf >= THRESHOLD:
                if not sentence or sentence[-1] != pred_word:
                    sentence.append(pred_word)
                    print(f'Added: {pred_word}  |  Sentence: {sentence}')
            elif key == ord('c'):
                sentence.clear()

    cap.release()
    cv2.destroyAllWindows()
    print('Final sentence:', ' '.join(sentence) if sentence else '(empty)')

print('run_detector() is ready.')

run_detector() is ready.


In [13]:
# Run the webcam detector
# Change camera_id=1 if the default camera does not open
run_detector(camera_id=0)

Webcam open. Press A:Add  C:Clear  Q:Quit
Final sentence: (empty)


In [12]:
REALTIME_SCRIPT = "import os, json, time, collections\nimport numpy as np\nimport cv2\nimport mediapipe as mp\nfrom tensorflow.keras.models import load_model\nfrom mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions\nfrom mediapipe.tasks.python.vision.hand_landmarker import HandLandmarksConnections\nfrom mediapipe.tasks.python.vision import drawing_utils  as mp_drawing\nfrom mediapipe.tasks.python.vision import drawing_styles as mp_draw_styles\nfrom mediapipe.tasks.python.core.base_options import BaseOptions\nfrom mediapipe.tasks.python.vision.core.vision_task_running_mode import VisionTaskRunningMode\n\nmodel       = load_model('models/best_lstm_model.keras')\nmeta        = json.load(open('models/model_metadata.json'))\ncfg         = json.load(open('processed_data/sequence_config.json'))\nCLASS_NAMES = meta['class_names']\nSEQ_LEN     = cfg['SEQUENCE_LENGTH']\nFEAT_SIZE   = cfg['FEATURE_SIZE']\nTHRESHOLD   = meta.get('confidence_threshold', 0.7)\nHAND_CONN   = HandLandmarksConnections.HAND_CONNECTIONS\n\ndef make_landmarker():\n    opts = HandLandmarkerOptions(\n        base_options=BaseOptions(model_asset_path='hand_landmarker.task'),\n        running_mode=VisionTaskRunningMode.VIDEO,\n        num_hands=2, min_hand_detection_confidence=0.5,\n        min_hand_presence_confidence=0.5, min_tracking_confidence=0.5,\n    )\n    return HandLandmarker.create_from_options(opts)\n\ndef get_landmarks(frame, landmarker, ts_ms):\n    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n    result = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts_ms)\n    lm, found, ann = np.zeros(FEAT_SIZE, np.float32), False, frame.copy()\n    if result.hand_landmarks:\n        found = True\n        for hi, hand in enumerate(result.hand_landmarks[:2]):\n            mp_drawing.draw_landmarks(ann, hand, HAND_CONN,\n                mp_draw_styles.get_default_hand_landmarks_style(),\n                mp_draw_styles.get_default_hand_connections_style())\n            off = hi * 63\n            for li, l in enumerate(hand):\n                lm[off + li*3], lm[off + li*3+1], lm[off + li*3+2] = l.x, l.y, l.z\n    return lm, found, ann\n\ndef run_detector(camera_id=0):\n    cap = cv2.VideoCapture(camera_id)\n    if not cap.isOpened():\n        print(f'Cannot open camera {camera_id}')\n        return\n    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)\n    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)\n\n    buffer    = collections.deque(maxlen=SEQ_LEN)\n    prob_hist = collections.deque(maxlen=10)\n    sentence  = []\n    fps_q     = collections.deque(maxlen=30)\n    t0        = time.time()\n    t0_ms     = int(t0 * 1000)\n    prev_t    = t0\n    last_hand = False\n    pred_word, pred_conf = None, 0.0\n\n    with make_landmarker() as lm_detector:\n        while True:\n            ret, frame = cap.read()\n            if not ret:\n                break\n            frame  = cv2.flip(frame, 1)\n            ts_ms  = int(time.time() * 1000) - t0_ms\n            lm_vec, hand_found, ann = get_landmarks(frame, lm_detector, ts_ms)\n\n            if last_hand and not hand_found:\n                buffer.clear()\n                prob_hist.clear()\n            last_hand = hand_found\n            buffer.append(lm_vec)\n\n            if len(buffer) == SEQ_LEN:\n                seq   = np.expand_dims(np.array(buffer, np.float32), 0)\n                probs = model.predict(seq, verbose=0)[0]\n                prob_hist.append(probs)\n                avg   = np.mean(list(prob_hist), axis=0)\n                idx   = int(np.argmax(avg))\n                pred_word, pred_conf = CLASS_NAMES[idx], float(avg[idx])\n\n            now = time.time()\n            fps_q.append(1.0 / max(now - prev_t, 1e-6))\n            prev_t = now\n            fps    = np.mean(fps_q)\n\n            h, w = ann.shape[:2]\n            cv2.rectangle(ann, (0, 0), (w, 60), (0, 0, 0), -1)\n            color = (0, 255, 0) if hand_found else (0, 0, 255)\n            cv2.putText(ann, 'HAND ON' if hand_found else 'HAND OFF',\n                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)\n            cv2.putText(ann, f'FPS: {fps:.1f}',\n                        (w - 120, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)\n\n            if pred_word and pred_conf >= THRESHOLD:\n                cv2.putText(ann, f'{pred_word}  {pred_conf*100:.0f}%',\n                            (10, h - 60), cv2.FONT_HERSHEY_DUPLEX, 1.5, (0, 255, 100), 3)\n            if sentence:\n                cv2.putText(ann, 'Sentence: ' + ' '.join(sentence[-6:]),\n                            (10, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)\n            cv2.putText(ann, 'A:Add  C:Clear  Q:Quit',\n                        (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)\n\n            cv2.imshow('ASL Real-Time Detector', ann)\n            key = cv2.waitKey(1) & 0xFF\n            if key in (ord('q'), 27):\n                break\n            elif key == ord('a') and pred_word and pred_conf >= THRESHOLD:\n                if not sentence or sentence[-1] != pred_word:\n                    sentence.append(pred_word)\n                    print(f'Added: {pred_word}  |  Sentence: {sentence}')\n            elif key == ord('c'):\n                sentence.clear()\n\n    cap.release()\n    cv2.destroyAllWindows()\n    print('Final sentence:', ' '.join(sentence) if sentence else '(empty)')\n\nif __name__ == '__main__':\n    run_detector()\n"

# Save as a standalone Python script
with open('realtime_asl.py', 'w', encoding='utf-8') as f:
    f.write(REALTIME_SCRIPT)
print('Saved: realtime_asl.py')
print()
print('Run from terminal:  python realtime_asl.py')

Saved: realtime_asl.py

Run from terminal:  python realtime_asl.py
